In [1]:
import sys
from pathlib import Path

# Path.cwd() obtiene la carpeta actual (sandbox). .parent nos sube a la raíz del proyecto.
raiz = Path.cwd().parent

if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.esios_client import crear_sesion, descargar_indicador

In [2]:
from dotenv import load_dotenv
import pandas as pd
import os

load_dotenv()
# Si tu .env está en una carpeta superior o una ruta específica, puedes indicarla:
# load_dotenv(dotenv_path="../.env")

# Recuperamos el token de forma segura
TOKEN = os.getenv("API_ESIOS")

# Verificación rápida (sin mostrar el token entero por seguridad)
if TOKEN:
    print(f"✅ Token cargado correctamente. Longitud: {len(TOKEN)} caracteres.")
else:
    print("❌ No se pudo encontrar la variable ESIOS_TOKEN. Revisa la ruta del archivo .env.")

✅ Token cargado correctamente. Longitud: 64 caracteres.


In [3]:
sesion = crear_sesion(TOKEN)

In [4]:
inicio = "2024-01-01T00:00:00"
fin = "2024-01-31T23:59:59"
indicador_id = 1293
# 1293: Demanda Real Peninsular
# 544: Previsión de la Demanda de REE
url = f"https://api.esios.ree.es/indicators/{indicador_id}"
params = {"start_date": inicio, "end_date": fin}

# SOLUCIÓN ROBUSTA: Timeout como tupla (connect, read)
# 5s para establecer conexión (falla rápido si la API está caída)
# 60s para leer datos (da margen si el volumen de filas ralentiza la respuesta)
response = sesion.get(url, params=params, timeout=(5, 60))
response.raise_for_status()

# Nota de diseño: Si cambia el esquema del JSON, el KeyError resultante
# se captura y enriquece con contexto en la capa del runner superior.
datos_json = response.json()['indicator']['values']

df = pd.DataFrame(datos_json)

In [5]:
print(df.columns.tolist())


['value', 'datetime', 'datetime_utc', 'tz_time', 'geo_id', 'geo_name']


In [6]:
df.head()

,value,datetime,datetime_utc,tz_time,geo_id,geo_name
0,22021.0,2024-01-01T00:00:00.000+01:00,2023-12-31T23:00:00Z,2023-12-31T23:00:00.000Z,8741,Península
1,22062.0,2024-01-01T00:05:00.000+01:00,2023-12-31T23:05:00Z,2023-12-31T23:05:00.000Z,8741,Península
2,22101.0,2024-01-01T00:10:00.000+01:00,2023-12-31T23:10:00Z,2023-12-31T23:10:00.000Z,8741,Península
3,21991.0,2024-01-01T00:15:00.000+01:00,2023-12-31T23:15:00Z,2023-12-31T23:15:00.000Z,8741,Península
4,21893.0,2024-01-01T00:20:00.000+01:00,2023-12-31T23:20:00Z,2023-12-31T23:20:00.000Z,8741,Península


In [7]:
print(df['geo_id'].unique())

[8741]


In [8]:
inicio = pd.Timestamp("2024-03-01 00:00", tz="UTC")
fin = pd.Timestamp("2024-04-01 00:00", tz="UTC")
df = descargar_indicador(sesion, indicador_id, inicio, fin, "Demanda_Real")

In [9]:
df.head()

,datetime_utc,Demanda_Real
0,2024-03-01 00:00:00+00:00,24441.0
1,2024-03-01 00:05:00+00:00,24538.0
2,2024-03-01 00:10:00+00:00,24522.0
3,2024-03-01 00:15:00+00:00,24391.0
4,2024-03-01 00:20:00+00:00,24233.0


In [10]:
len(df)

8928

In [11]:
inicio = pd.Timestamp("2024-03-01 00:00", tz="UTC")
fin = pd.Timestamp("2024-04-01 00:00", tz="UTC")
df = descargar_indicador(sesion, indicador_id, inicio, fin, "Demanda_Real")
df

,datetime_utc,Demanda_Real
0,2024-03-01 00:00:00+00:00,24441.0
1,2024-03-01 00:05:00+00:00,24538.0
2,2024-03-01 00:10:00+00:00,24522.0
3,2024-03-01 00:15:00+00:00,24391.0
4,2024-03-01 00:20:00+00:00,24233.0
...,...,...
8923,2024-03-31 23:35:00+00:00,19898.0
8924,2024-03-31 23:40:00+00:00,19858.0
8925,2024-03-31 23:45:00+00:00,19713.0
8926,2024-03-31 23:50:00+00:00,19544.0


In [13]:
len(df)

8928

In [15]:
def comprobar_duplicados(df: pd.DataFrame, inicio: pd.Timestamp, fin: pd.Timestamp):
    """
    Gate de validación estricta para el DataFrame de ESIOS.
    Comprueba anclaje de inicio, regularidad de 5 min y longitud exacta.
    Lanza AssertionError si alguna condición no se cumple.
    """
    # 0. El DataFrame no puede estar vacío si se esperan datos
    assert not df.empty, "El DataFrame está completamente vacío."
    
    # Aseguramos que las entradas de control estén en UTC para comparar peras con peras
    inicio_utc = inicio.tz_convert('UTC')
    fin_utc = fin.tz_convert('UTC')
    
    # 1. CHECK ANCLAJE: Evita que la rejilla esté desplazada (el hueco que detectaste)
    assert df['datetime_utc'].iloc[0] == inicio_utc, \
        f"Error de anclaje: El primer registro ({df['datetime_utc'].iloc[0]}) no coincide con el inicio pedido ({inicio_utc})."
        
    # 2. CHECK DELTA: CAZA duplicados (delta 0) y huecos internos (delta > 5min)
    # Usamos .dropna() para ignorar el NaT de la primera fila que genera .diff()
    deltas = df['datetime_utc'].diff().dropna()
    assert deltas.eq(pd.Timedelta(minutes=5)).all(), \
        "Error de regularidad: Se detectaron saltos de tiempo incorrectos o filas duplicadas."
        
    # 3. CHECK LONGITUD: Ancla el span y asegura que no falten filas al inicio/final
    filas_esperadas = int((fin_utc - inicio_utc) / pd.Timedelta(minutes=5)) 
    
    assert len(df) == filas_esperadas, \
        f"Error de longitud: Se esperaban {filas_esperadas} filas, pero se obtuvieron {len(df)}."
    
comprobar_duplicados(df, inicio, fin)